# 03 — Harmonize corrected hyperspectral reflectance

Use this focused notebook after correction when you want the convolution and table-writing part of the full raster workflow. It uses canonical paths, prints the stage result, and inventories the sensor products it wrote.

## 1. Configure the corrected flightline

Use the same output root and flightline stem as notebook 02. `FlightlinePaths` resolves the naming contract rather than reconstructing corrected filenames by hand.

In [ ]:
from pathlib import Path
from pprint import pprint

from spectralbridge.paths import FlightlinePaths
from spectralbridge.pipelines.pipeline import stage_convolve_all_sensors

RUN = False
base_folder = Path("outputs/neon_notebook")
product_code = "DP1.30006.001"
flight_stem = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
paths = FlightlinePaths(base_folder, flight_stem)

## 2. Convolve and write analysis products

The stage reuses valid target-sensor products. Keep `resample_method="convolution"` for the documented sensor-response workflow and `extraction_mode="full"` for a full-pixel table.

In [ ]:
print(f"Corrected image: {paths.corrected_img}")
print(f"Corrected header: {paths.corrected_hdr}")
summary = None
if RUN:
    summary = stage_convolve_all_sensors(
        base_folder=base_folder,
        product_code=product_code,
        flight_stem=flight_stem,
        corrected_img_path=paths.corrected_img,
        corrected_hdr_path=paths.corrected_hdr,
        resample_method="convolution",
        extraction_mode="full",
    )
    pprint(summary)
else:
    print("Dry run. Set RUN = True after the corrected ENVI pair validates.")

## 3. Check sensor and table outputs

Inventory filenames and sizes before analysis. The package's path and naming utilities remain the authority for deciding whether a product is reusable.

In [ ]:
flight_dir = paths.flight_dir
products = sorted(flight_dir.glob("*")) if flight_dir.exists() else []
for path in products:
    if path.is_file() and path.suffix in {'.img', '.hdr', '.parquet', '.csv'}:
        print(f"{path.name}: {path.stat().st_size:,} bytes")
if not products:
    print("No flightline products found. Complete notebook 02 first.")

## 4. Continue

Use notebook 04 to query the merged Parquet table and notebook 05 to compare tabular results with the QA imagery. Landsat is the target bandspace here; the translation is mediated by the corrected NEON hyperspectral representation.